# Exp #6 — Guidance-Gap generation on Kaggle (CUDA, fp16)

Generates the 264-image guidance-gap set (24 prompts × 1 seed × [1 standard + 10 γ₂ variants]) for the Z-Sampling guidance-gap ablation.

Why Kaggle: on CUDA the model runs in **fp16** (paper-correct), avoiding the Apple-MPS fp32 penalty. Estimated **~6–9h** on a P100 — fits the 30 GPU-h/week quota.

**Before you run, set notebook settings (right panel):**
- **Accelerator: GPU P100** (the script is single-GPU; P100 beats T4×2 here).
- **Internet: ON** (needed to download SDXL-base ≈ 7 GB from HuggingFace).
- **Persistence: "Files only"** — keeps `/kaggle/working` across sessions so a timeout-then-relaunch **resumes** instead of restarting.

Scoring is done **locally on the Mac** afterward — this notebook only generates images and packages them for download.

## 1. Clone the code (idempotent)
Requires the `radu-exp-6` branch to be pushed to GitHub first (see RUNBOOK §0). If the repo is **private**, add a GitHub token as a Kaggle Secret named `GH_TOKEN` and the clone below will use it.

In [ ]:
import os
import pathlib
import subprocess

REPO_DIR = pathlib.Path('/kaggle/working/gm')
BRANCH = 'radu-exp-6'

# Optional private-repo token via Kaggle Secrets (Add-ons → Secrets → GH_TOKEN)
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    pass
auth = f'{token}@' if token else ''
url = f'https://{auth}github.com/viftode4/generative-modelling.git'

if (REPO_DIR / '.git').exists():
    print('repo present — pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--depth', '1', url, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=False)

## 2. Install generation deps
Kaggle's preinstalled CUDA-enabled `torch` is kept as-is; we add only the diffusion stack the validated env used. (Scoring deps are NOT installed — scoring runs on the Mac.)

In [ ]:
!pip install -q \
  diffusers==0.37.1 transformers==5.8.1 accelerate safetensors einops torchsde \
  'huggingface_hub>=0.20' 'numpy>=2,<3' tqdm Pillow

In [ ]:
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable GPU accelerator in notebook settings!'

## 3. Run generation (resumable)
Writes images + `manifest.jsonl` to `outputs/radu_guidance_gap/kaggle_24x1`. The script **skips any image already on disk**, so if the session times out, just rerun this notebook (with Persistence on) and it continues.

Defaults match the paper: 1024px, 50 steps, λ-step 49, γ₁=5.5, 10-point γ₂ sweep.

In [ ]:
OUT = 'outputs/radu_guidance_gap/kaggle_24x1'

# Dry-run first: prints the full job plan / image count without loading the model.
!python scripts/run_radu_guidance_gap.py --out {OUT} --device cuda --dry-run | tail -5

In [ ]:
# The real run. ~6-9h on P100. Safe to interrupt/relaunch (resumes from disk).
!python scripts/run_radu_guidance_gap.py \
  --out outputs/radu_guidance_gap/kaggle_24x1 \
  --device cuda --size 1024 --steps 50 --lambda-step 49 --gamma1 5.5

## 4. Package for download → score on Mac
Bundles images + manifest into one tarball in `/kaggle/working` (download via the Output panel). On the Mac, untar into the repo's `outputs/` and run the local scoring scripts.

In [ ]:
import glob
import subprocess

n = len(glob.glob('outputs/radu_guidance_gap/kaggle_24x1/images/*.png'))
print(f'{n} images generated (expect 264 when complete)')
subprocess.run(
    ['tar', 'czf', '/kaggle/working/guidance_gap_kaggle_24x1.tgz',
     '-C', 'outputs/radu_guidance_gap', 'kaggle_24x1'], check=True)
print('wrote /kaggle/working/guidance_gap_kaggle_24x1.tgz')
subprocess.run(['ls', '-lh', '/kaggle/working/guidance_gap_kaggle_24x1.tgz'])

## 5. If the run didn't finish in one session
With **Persistence = Files only**, `/kaggle/working` (incl. the cloned repo and generated images) survives. Just **Run All** again — cell 3 skips done images and continues. Repeat across sessions until cell 4 reports 264 images, then download the tarball.

On the Mac:
```bash
tar xzf ~/Downloads/guidance_gap_kaggle_24x1.tgz -C generative-modelling/outputs/radu_guidance_gap/
# then run the local scoring pipeline (see RUNBOOK §4)
```